# KLA PS01 -- Restoration Model Training (Kaggle)

Trains the NAFNet-lite denoise+2x-SR model on KLA's paired semiconductor
inspection dataset, on Kaggle's free GPU instead of Colab.

**Why Kaggle instead of Colab:** Colab's free tier throttles you on GPU
*compute units* (can cut you off well before a session's 12h wall-clock
cap, and needs the browser tab open). Kaggle gives **30 GPU-hours/week**
and lets you commit a notebook to run in the background for up to **9
hours per session** even with the browser closed (Save Version -> Save
& Run All). With the mixed-precision (AMP) speedup in `train.py`,
100 epochs should take well under 2 hours on a T4/P100, so this should
finish in a single session.

**Before running, in the notebook's right sidebar (Settings):**
1. **Accelerator:** GPU T4 x2 or GPU P100 (either works -- this script
   only uses a single GPU).
2. **Internet:** On (needed for `git clone`, `pip install`, and the
   VGG16 pretrained-weights download). Requires a phone-verified Kaggle
   account.

**You need to upload the dataset first**, as a Kaggle Dataset:
kaggle.com -> Create -> New Dataset -> upload your local `train/`
folder (or a zip of it, containing `GT/` and `NoisyLR/` subfolders of
`.npy` files) -- then "Add Data" on this notebook to attach it. Update
`DATASET_DIR` in the "Locate dataset" cell below to match.

## 1. Check GPU

In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Clone the repo

Pulls the latest `src/` (dataset.py, model.py, losses.py, train.py, inference.py) straight from GitHub -- no need to duplicate code into notebook cells like the Colab version does.

In [ ]:
import os

REPO_DIR = "/kaggle/working/semicon"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/Ganesh-0509/semicon.git "{REPO_DIR}"
else:
    !cd "{REPO_DIR}" && git pull

## 3. Install extra dependencies

Kaggle's GPU image already ships torch/torchvision/numpy/Pillow with CUDA wired up -- only the extras used by the metrics cell need installing.

In [ ]:
!pip install -q lpips scikit-image

## 4. Locate dataset

Edit `DATASET_DIR` to point at the Kaggle Dataset you attached (visible
under `/kaggle/input/`). Handles both cases: Kaggle auto-extracted your
zip into `GT/`/`NoisyLR/` folders already, or the zip is still sitting
there unextracted.

In [ ]:
import os, zipfile, shutil

DATASET_DIR = "/kaggle/input/kla-ps01-train"  # <-- EDIT THIS to your attached dataset's slug
EXTRACT_ROOT = "/kaggle/working/data"

assert os.path.isdir(DATASET_DIR), (
    f"{DATASET_DIR} not found -- check the exact slug under /kaggle/input/ "
    f"(left sidebar -> Data) and fix DATASET_DIR above."
)


def find_gt_noisy(root):
    for r, dirs, files in os.walk(root):
        if "GT" in dirs and "NoisyLR" in dirs:
            return r
    return None


DATA_ROOT = find_gt_noisy(DATASET_DIR)

if DATA_ROOT is None:
    # Not auto-extracted -- look for a zip and extract it ourselves.
    zips = [os.path.join(r, f) for r, _, fs in os.walk(DATASET_DIR) for f in fs if f.lower().endswith(".zip")]
    assert zips, f"No GT/NoisyLR folders and no .zip found under {DATASET_DIR}."
    zip_path = zips[0]
    if os.path.isdir(EXTRACT_ROOT):
        shutil.rmtree(EXTRACT_ROOT)
    os.makedirs(EXTRACT_ROOT, exist_ok=True)
    BACKSLASH = chr(92)
    with zipfile.ZipFile(zip_path, "r") as zf:
        for info in zf.infolist():
            norm_name = info.filename.replace(BACKSLASH, "/")
            if norm_name.endswith("/"):
                continue
            dest_path = os.path.join(EXTRACT_ROOT, *norm_name.split("/"))
            os.makedirs(os.path.dirname(dest_path), exist_ok=True)
            with zf.open(info) as src, open(dest_path, "wb") as dst:
                shutil.copyfileobj(src, dst)
    DATA_ROOT = find_gt_noisy(EXTRACT_ROOT)

assert DATA_ROOT is not None, "Could not locate GT/NoisyLR anywhere -- inspect the dataset structure manually."
print("Using DATA_ROOT =", DATA_ROOT)
print("GT files:", len(os.listdir(os.path.join(DATA_ROOT, "GT"))))
print("NoisyLR files:", len(os.listdir(os.path.join(DATA_ROOT, "NoisyLR"))))

## 5. Sanity check: smoke test on GPU (a few steps only)

In [ ]:
!cd "{REPO_DIR}/src" && python train.py --smoke_test --device cuda --data_root "{DATA_ROOT}"

## 6. Full training run

`--out_dir` points at `/kaggle/working/checkpoints` -- files there
persist for the life of this session, and become a downloadable output
once you commit (Save Version -> Save & Run All). `--num_workers 2`
matches Kaggle's CPU allocation (4 tends to stall the dataloader, same
issue as on Colab). `train.py` now uses mixed precision automatically
on CUDA, so this should run noticeably faster than the Colab run did.

If the session gets interrupted mid-run, re-run this same cell after
uncommenting the `--resume` line to continue from the last saved epoch
instead of starting over -- `resume.pt` is written to `--out_dir` after
every epoch.

In [ ]:
CKPT_DIR = "/kaggle/working/checkpoints"
!mkdir -p "{CKPT_DIR}"
!cd "{REPO_DIR}/src" && python train.py \
    --data_root "{DATA_ROOT}" \
    --out_dir "{CKPT_DIR}" \
    --epochs 100 \
    --batch_size 16 \
    --lr 2e-4 \
    --device cuda \
    --num_workers 2 \
    --use_perceptual
    # --resume "{CKPT_DIR}/resume.pt"   # <-- uncomment to continue after an interruption

## 7. Resuming across separate Kaggle sessions

`/kaggle/working` only survives within one running session unless you
commit. If you need more than one session to finish training:

1. Before the session ends, click **Save Version -> Save & Run All
   (Commit)**. The `checkpoints/` folder (including `resume.pt`) becomes
   a downloadable **Output** of that notebook version.
2. Start a new session. Under **Add Data**, attach *this same notebook's
   previous version output* as an input (Kaggle lists your own notebook
   outputs alongside datasets).
3. Point `--resume` at wherever that output landed under
   `/kaggle/input/<this-notebook-slug>/checkpoints/resume.pt`, and keep
   `--epochs 100` unchanged (the cosine LR schedule was built against
   `T_max=100` -- changing it on resume desyncs the schedule).

## 8. Verify a checkpoint loads cleanly

In [ ]:
import torch, sys
sys.path.insert(0, f"{REPO_DIR}/src")
from model import build_model
m = build_model()
state = torch.load("/kaggle/working/checkpoints/best.pt", map_location="cpu")
m.load_state_dict(state)
print("Checkpoint loads cleanly. Params:", sum(p.numel() for p in m.parameters()))

## 9. Run inference on the held-out test set + benchmark speed

Point `TEST_INPUT_DIR` at your uploaded held-out `NoisyLR/`-style test
folder (no GT) -- either a separate attached dataset, or a subfolder of
the same one. This mirrors exactly what KLA will run against your repo.

In [ ]:
TEST_INPUT_DIR = os.path.join(DATA_ROOT, "NoisyLR")  # <-- EDIT if using a separate held-out test dataset
TEST_OUTPUT_DIR = "/kaggle/working/outputs"

!cd "{REPO_DIR}/src" && python inference.py "{TEST_INPUT_DIR}" "{TEST_OUTPUT_DIR}" \
    --weights /kaggle/working/checkpoints/best.pt --device cuda

## 10. Compute SSIM / PSNR / LPIPS on your own validation split

(For the same slide/report numbers as the Colab notebook. Requires
ground truth, so this runs against your local val split, not the blind
test set.)

In [ ]:
import torch, numpy as np, lpips, sys
from skimage.metrics import structural_similarity as ssim_fn
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
sys.path.insert(0, f"{REPO_DIR}/src")
from dataset import make_splits, SubsetRestorationDataset
from model import build_model
from torch.utils.data import DataLoader

device = "cuda"
gt_dir = DATA_ROOT + "/GT"
noisy_dir = DATA_ROOT + "/NoisyLR"
_, val_ids = make_splits(gt_dir, noisy_dir, val_fraction=0.1, seed=42)
val_ds = SubsetRestorationDataset(gt_dir, noisy_dir, val_ids, train=False)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)

model = build_model().to(device)
model.load_state_dict(torch.load("/kaggle/working/checkpoints/best.pt", map_location=device))
model.eval()

lpips_fn = lpips.LPIPS(net="alex").to(device)

ssims, psnrs, lpipss = [], [], []
with torch.no_grad():
    for batch in val_loader:
        noisy = batch["noisy_lr"].to(device)
        gt = batch["gt"].to(device)
        pred = model(noisy).clamp(0, 1)

        lp = lpips_fn(pred * 2 - 1, gt * 2 - 1).squeeze().cpu().numpy()
        lpipss.extend(np.atleast_1d(lp).tolist())

        pred_np = pred.squeeze(1).cpu().numpy()
        gt_np = gt.squeeze(1).cpu().numpy()
        for p, g in zip(pred_np, gt_np):
            ssims.append(ssim_fn(g, p, data_range=1.0))
            psnrs.append(psnr_fn(g, p, data_range=1.0))

print(f"SSIM:  {np.mean(ssims):.4f}")
print(f"PSNR:  {np.mean(psnrs):.2f} dB")
print(f"LPIPS: {np.mean(lpipss):.4f}")